# Visualize Normalized Abundances

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from hk1_r12ter_analysis.config import (
    EXTERNAL_DATA_DIR,
    PROCESSED_DATA_DIR,
    REPORTS_DIR,
)
from hk1_r12ter_analysis.io import (
    load_dataframe_from_file,
    make_filename,
    save_dataframe_to_file,
)

## Load data

In [ ]:
sample_key = "Sample"
group_key = "Group"
group1 = "Control"
group2 = "HK1"

normalization = "median-none-auto"

In [ ]:
def plot_abundances(data, x, y, ax=None, **kwargs):
    """Plot normalized abundance as individual data points, means, and error bars."""
    linewidth = kwargs.get("linewidth", 1)
    min_zorder = kwargs.get("min_zorder", 3)
    orient = kwargs.get("orient", "v")
    palette = kwargs.get("palette")

    pointplot_kwargs = kwargs.get("pointplot_kwargs", {})
    stripplot_kwargs = kwargs.get("stripplot_kwargs", {})
    legend_kwargs = kwargs.get("legend_kwargs", {})

    ax = sns.pointplot(
        data=data,
        x=x,
        y=y,
        ax=ax,
        hue=x,
        palette=palette,
        linestyle="None",
        markersize=pointplot_kwargs.get("markersize", 8),
        # markeredgecolor=pointplot_kwargs.get("markeredgecolor", "xkcd:black"),
        markeredgewidth=pointplot_kwargs.get("markeredgewidth", 1),
        zorder=pointplot_kwargs.get("zorder", min_zorder + 1),
        estimator=pointplot_kwargs.get("estimator", "mean"),
        errorbar=pointplot_kwargs.get("errorbar", "sd"),
        err_kws=pointplot_kwargs.get(
            "err_kws",
            dict(
                zorder=min_zorder,
                linewidth=pointplot_kwargs.get("linewidth", 2),
            ),
        ),
        capsize=pointplot_kwargs.get("capsize", 0.4),
    )
    np.random.seed(stripplot_kwargs.get("seed", 4))
    ax = sns.stripplot(
        data=data,
        x=x,
        y=y,
        ax=ax,
        jitter=False,
        zorder=stripplot_kwargs.get("zorder", min_zorder + 2),
        marker=stripplot_kwargs.get("marker", "D"),
        s=stripplot_kwargs.get("s", 8),
        c=stripplot_kwargs.get("c", "xkcd:grey"),
        edgecolor=stripplot_kwargs.get("edgecolor", "xkcd:black"),
        linewidth=stripplot_kwargs.get("linewidth", 1),
    )

    return ax

### Common keyword arguments

In [ ]:
orient = "v"
linewidth = 1
fontdict = dict(size="large")
min_zorder = 3

pointplot_kwargs = dict(markersize=15, linewidth=5, edgecolor="black")
stripplot_kwargs = dict(s=5, seed=5, c="xkcd:grey")
legend_kwargs = dict()

### RBC data

#### Metabolites

In [ ]:
source_type = "RBC"
specie_type = "Metabolites"

input_filename = make_filename(source_type, specie_type, "Intensities")
input_dirpath = PROCESSED_DATA_DIR / "HK1-R12Ter" / "Normalized" / normalization

output_filetype = "pdf"
output_filename = make_filename(source_type, specie_type, "AbundanceStats")
output_dirpath = (
    REPORTS_DIR
    / "HK1-R12Ter"
    / "Normalized"
    / normalization
    / f"{source_type}-NormalizedAbundance-MeanSEM-{specie_type}"
)
output_dirpath.mkdir(parents=True, exist_ok=True)

df_abundance = load_dataframe_from_file(
    input_dirpath / input_filename, index_col=[sample_key, group_key]
)

input_filename = make_filename(source_type, specie_type, "t_test_neq", filetype="tsv")
input_dirpath = EXTERNAL_DATA_DIR / "MetaboAnalyst" / "HK1-R12Ter-Statistics"
df_pvalues = load_dataframe_from_file(input_dirpath / input_filename, index_col=0)
df_pvalues = df_pvalues["p-value"]
# Plot metabolites with p < 0.1, provides some flexibility
cutoff = 0.1
species_to_plot = sorted(list(df_pvalues[df_pvalues <= cutoff].index))

for variable in species_to_plot:
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(2.5, 5))
    data = df_abundance.loc[:, [variable]]
    ax = plot_abundances(
        data=data,
        x=variable if orient == "h" else group_key,
        y=group_key if orient == "h" else variable,
        ax=ax,
        **dict(
            orient=orient,
            palette={
                group1: "#339966",
                group2: "#cc6633",
            },
            pointplot_kwargs=pointplot_kwargs,
            stripplot_kwargs=stripplot_kwargs,
            legend_kwargs=legend_kwargs,
        ),
    )

    ax.set_xlabel(group_key, fontdict=fontdict)
    ax.xaxis.grid(True, which="both")
    ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(1.5))
    ax.yaxis.set_minor_locator(mpl.ticker.MultipleLocator(0.5))
    ax.yaxis.set_tick_params(which="both", length=10)
    ax.set_ylim((-1.75, 1.75))
    ax.set_ylabel("Normalized Abundance", fontdict=fontdict)
    ax.yaxis.grid(True, which="both")
    ax.set_title(f"{variable} p={df_pvalues.loc[variable]:.4f}", fontdict=fontdict)
    sns.despine(fig)

    output_filename = make_filename(
        variable.replace("/", "-or-").replace(":", "_").replace(",", "-"), filetype=output_filetype
    )
    fig.savefig(output_dirpath / output_filename)
    plt.close()

df = df_abundance.copy()
df = df.groupby(group_key).agg(("mean", "std", "sem")).T
df.index.names = [specie_type, "Value"]
# df.columns.name = None
df = df.reset_index(drop=False)
df = df.pivot(columns=["Value"], index=[specie_type])

output_filename = make_filename(source_type, specie_type, "AbundanceStats")
save_dataframe_to_file(df, output_dirpath / output_filename, index=True)
df

#### Proteins

In [ ]:
source_type = "RBC"
specie_type = "Proteins"

input_filename = make_filename(source_type, specie_type, "Intensities")
input_dirpath = PROCESSED_DATA_DIR / "HK1-R12Ter" / "Normalized" / normalization

output_filetype = "pdf"
output_filename = make_filename(source_type, specie_type, "AbundanceStats")
output_dirpath = (
    REPORTS_DIR
    / "HK1-R12Ter"
    / "Normalized"
    / normalization
    / f"{source_type}-NormalizedAbundance-MeanSEM-{specie_type}"
)
output_dirpath.mkdir(parents=True, exist_ok=True)

df_abundance = load_dataframe_from_file(
    input_dirpath / input_filename, index_col=[sample_key, group_key]
)

input_filename = make_filename(source_type, specie_type, "t_test_neq", filetype="tsv")
input_dirpath = EXTERNAL_DATA_DIR / "MetaboAnalyst" / "HK1-R12Ter-Statistics"
df_pvalues = load_dataframe_from_file(input_dirpath / input_filename, index_col=0)
df_pvalues = df_pvalues["p-value"]
# Plot metabolites with p < 0.1, provides some flexibility
cutoff = 0.1
species_to_plot = sorted(list(df_pvalues[df_pvalues <= cutoff].index))

for variable in species_to_plot:
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(2.5, 5))
    data = df_abundance.loc[:, [variable]]
    ax = plot_abundances(
        data=data,
        x=variable if orient == "h" else group_key,
        y=group_key if orient == "h" else variable,
        ax=ax,
        **dict(
            orient=orient,
            palette={
                group1: "#339966",
                group2: "#cc6633",
            },
            pointplot_kwargs=pointplot_kwargs,
            stripplot_kwargs=stripplot_kwargs,
            legend_kwargs=legend_kwargs,
        ),
    )

    ax.set_xlabel(group_key, fontdict=fontdict)
    ax.xaxis.grid(True, which="both")
    ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(1.5))
    ax.yaxis.set_minor_locator(mpl.ticker.MultipleLocator(0.5))
    ax.yaxis.set_tick_params(which="both", length=10)
    ax.set_ylim((-1.75, 1.75))
    ax.set_ylabel("Normalized Abundance", fontdict=fontdict)
    ax.yaxis.grid(True, which="both")
    ax.set_title(f"{variable} p={df_pvalues.loc[variable]:.4f}", fontdict=fontdict)
    sns.despine(fig)

    output_filename = make_filename(
        variable.replace("/", "-or-").replace(":", "_").replace(",", "-"), filetype=output_filetype
    )
    fig.savefig(output_dirpath / output_filename)
    plt.close()

df = df_abundance.copy()
df = df.groupby(group_key).agg(("mean", "std", "sem")).T
df.index.names = [specie_type, "Value"]
# df.columns.name = None
df = df.reset_index(drop=False)
df = df.pivot(columns=["Value"], index=[specie_type])

output_filename = make_filename(source_type, specie_type, "AbundanceStats")
save_dataframe_to_file(df, output_dirpath / output_filename, index=True)
df

#### Lipids

In [ ]:
source_type = "RBC"
specie_type = "CombinedLipids"

input_filename = make_filename(source_type, specie_type, "Intensities")
input_dirpath = PROCESSED_DATA_DIR / "HK1-R12Ter" / "Normalized" / normalization

output_filetype = "pdf"
output_filename = make_filename(source_type, specie_type, "AbundanceStats")
output_dirpath = (
    REPORTS_DIR
    / "HK1-R12Ter"
    / "Normalized"
    / normalization
    / f"{source_type}-NormalizedAbundance-MeanSEM-{specie_type}"
)
output_dirpath.mkdir(parents=True, exist_ok=True)

df_abundance = load_dataframe_from_file(
    input_dirpath / input_filename, index_col=[sample_key, group_key]
)

input_filename = make_filename(source_type, specie_type, "t_test_neq", filetype="tsv")
input_dirpath = EXTERNAL_DATA_DIR / "MetaboAnalyst" / "HK1-R12Ter-Statistics"
df_pvalues = load_dataframe_from_file(input_dirpath / input_filename, index_col=0)
df_pvalues = df_pvalues["p-value"]
# Plot metabolites with p < 0.1, provides some flexibility
cutoff = 0.1
species_to_plot = sorted(list(df_pvalues[df_pvalues <= cutoff].index))

for variable in species_to_plot:
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(2.5, 5))
    data = df_abundance.loc[:, [variable]]
    ax = plot_abundances(
        data=data,
        x=variable if orient == "h" else group_key,
        y=group_key if orient == "h" else variable,
        ax=ax,
        **dict(
            orient=orient,
            palette={
                group1: "#339966",
                group2: "#cc6633",
            },
            pointplot_kwargs=pointplot_kwargs,
            stripplot_kwargs=stripplot_kwargs,
            legend_kwargs=legend_kwargs,
        ),
    )

    ax.set_xlabel(group_key, fontdict=fontdict)
    ax.xaxis.grid(True, which="both")
    ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(1.5))
    ax.yaxis.set_minor_locator(mpl.ticker.MultipleLocator(0.5))
    ax.yaxis.set_tick_params(which="both", length=10)
    ax.set_ylim((-1.75, 1.75))
    ax.set_ylabel("Normalized Abundance", fontdict=fontdict)
    ax.yaxis.grid(True, which="both")
    ax.set_title(f"{variable} p={df_pvalues.loc[variable]:.4f}", fontdict=fontdict)
    sns.despine(fig)

    output_filename = make_filename(
        variable.replace("/", "-or-").replace(":", "_").replace(",", "-"), filetype=output_filetype
    )
    fig.savefig(output_dirpath / output_filename)
    plt.close()

df = df_abundance.copy()
df = df.groupby(group_key).agg(("mean", "std", "sem")).T
df.index.names = [specie_type, "Value"]
# df.columns.name = None
df = df.reset_index(drop=False)
df = df.pivot(columns=["Value"], index=[specie_type])

output_filename = make_filename(source_type, specie_type, "AbundanceStats")
save_dataframe_to_file(df, output_dirpath / output_filename, index=True)
df

### Plasma data
#### Metabolites

In [ ]:
source_type = "Plasma"
specie_type = "Metabolites"

input_filename = make_filename(source_type, specie_type, "Intensities")
input_dirpath = PROCESSED_DATA_DIR / "HK1-R12Ter" / "Normalized" / normalization

output_filetype = "pdf"
output_filename = make_filename(source_type, specie_type, "AbundanceStats")
output_dirpath = (
    REPORTS_DIR
    / "HK1-R12Ter"
    / "Normalized"
    / normalization
    / f"{source_type}-NormalizedAbundance-MeanSEM-{specie_type}"
)
output_dirpath.mkdir(parents=True, exist_ok=True)

df_abundance = load_dataframe_from_file(
    input_dirpath / input_filename, index_col=[sample_key, group_key]
)

input_filename = make_filename(source_type, specie_type, "t_test_neq", filetype="tsv")
input_dirpath = EXTERNAL_DATA_DIR / "MetaboAnalyst" / "HK1-R12Ter-Statistics"
df_pvalues = load_dataframe_from_file(input_dirpath / input_filename, index_col=0)
df_pvalues = df_pvalues["p-value"]
# Plot metabolites with p < 0.1, provides some flexibility
cutoff = 0.1
species_to_plot = sorted(list(df_pvalues[df_pvalues <= cutoff].index))

for variable in species_to_plot:
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(2.5, 5))
    data = df_abundance.loc[:, [variable]]
    ax = plot_abundances(
        data=data,
        x=variable if orient == "h" else group_key,
        y=group_key if orient == "h" else variable,
        ax=ax,
        **dict(
            orient=orient,
            palette={
                group1: "#339966",
                group2: "#cc6633",
            },
            pointplot_kwargs=pointplot_kwargs,
            stripplot_kwargs=stripplot_kwargs,
            legend_kwargs=legend_kwargs,
        ),
    )

    ax.set_xlabel(group_key, fontdict=fontdict)
    ax.xaxis.grid(True, which="both")
    ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(1.5))
    ax.yaxis.set_minor_locator(mpl.ticker.MultipleLocator(0.5))
    ax.yaxis.set_tick_params(which="both", length=10)
    ax.set_ylim((-1.75, 1.75))
    ax.set_ylabel("Normalized Abundance", fontdict=fontdict)
    ax.yaxis.grid(True, which="both")
    ax.set_title(f"{variable} p={df_pvalues.loc[variable]:.4f}", fontdict=fontdict)
    sns.despine(fig)

    output_filename = make_filename(
        variable.replace("/", "-or-").replace(":", "_").replace(",", "-"), filetype=output_filetype
    )
    fig.savefig(output_dirpath / output_filename)
    plt.close()

df = df_abundance.copy()
df = df.groupby(group_key).agg(("mean", "std", "sem")).T
df.index.names = [specie_type, "Value"]
# df.columns.name = None
df = df.reset_index(drop=False)
df = df.pivot(columns=["Value"], index=[specie_type])

output_filename = make_filename(source_type, specie_type, "AbundanceStats")
save_dataframe_to_file(df, output_dirpath / output_filename, index=True)
df

#### Proteins

In [ ]:
source_type = "Plasma"
specie_type = "Proteins"

input_filename = make_filename(source_type, specie_type, "Intensities")
input_dirpath = PROCESSED_DATA_DIR / "HK1-R12Ter" / "Normalized" / normalization

output_filetype = "pdf"
output_filename = make_filename(source_type, specie_type, "AbundanceStats")
output_dirpath = (
    REPORTS_DIR
    / "HK1-R12Ter"
    / "Normalized"
    / normalization
    / f"{source_type}-NormalizedAbundance-MeanSEM-{specie_type}"
)
output_dirpath.mkdir(parents=True, exist_ok=True)

df_abundance = load_dataframe_from_file(
    input_dirpath / input_filename, index_col=[sample_key, group_key]
)

input_filename = make_filename(source_type, specie_type, "t_test_neq", filetype="tsv")
input_dirpath = EXTERNAL_DATA_DIR / "MetaboAnalyst" / "HK1-R12Ter-Statistics"
df_pvalues = load_dataframe_from_file(input_dirpath / input_filename, index_col=0)
df_pvalues = df_pvalues["p-value"]
# Plot metabolites with p < 0.1, provides some flexibility
cutoff = 0.1
species_to_plot = sorted(list(df_pvalues[df_pvalues <= cutoff].index))

for variable in species_to_plot:
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(2.5, 5))
    data = df_abundance.loc[:, [variable]]
    ax = plot_abundances(
        data=data,
        x=variable if orient == "h" else group_key,
        y=group_key if orient == "h" else variable,
        ax=ax,
        **dict(
            orient=orient,
            palette={
                group1: "#339966",
                group2: "#cc6633",
            },
            pointplot_kwargs=pointplot_kwargs,
            stripplot_kwargs=stripplot_kwargs,
            legend_kwargs=legend_kwargs,
        ),
    )

    ax.set_xlabel(group_key, fontdict=fontdict)
    ax.xaxis.grid(True, which="both")
    ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(1.5))
    ax.yaxis.set_minor_locator(mpl.ticker.MultipleLocator(0.5))
    ax.yaxis.set_tick_params(which="both", length=10)
    ax.set_ylim((-1.75, 1.75))
    ax.set_ylabel("Normalized Abundance", fontdict=fontdict)
    ax.yaxis.grid(True, which="both")
    ax.set_title(f"{variable} p={df_pvalues.loc[variable]:.4f}", fontdict=fontdict)
    sns.despine(fig)

    output_filename = make_filename(
        variable.replace("/", "-or-").replace(":", "_").replace(",", "-"), filetype=output_filetype
    )
    fig.savefig(output_dirpath / output_filename)
    plt.close()

df = df_abundance.copy()
df = df.groupby(group_key).agg(("mean", "std", "sem")).T
df.index.names = [specie_type, "Value"]
# df.columns.name = None
df = df.reset_index(drop=False)
df = df.pivot(columns=["Value"], index=[specie_type])

output_filename = make_filename(source_type, specie_type, "AbundanceStats")
save_dataframe_to_file(df, output_dirpath / output_filename, index=True)
df

#### Lipids

In [ ]:
source_type = "Plasma"
specie_type = "CombinedLipids"

input_filename = make_filename(source_type, specie_type, "Intensities")
input_dirpath = PROCESSED_DATA_DIR / "HK1-R12Ter" / "Normalized" / normalization

output_filetype = "pdf"
output_filename = make_filename(source_type, specie_type, "AbundanceStats")
output_dirpath = (
    REPORTS_DIR
    / "HK1-R12Ter"
    / "Normalized"
    / normalization
    / f"{source_type}-NormalizedAbundance-MeanSEM-{specie_type}"
)
output_dirpath.mkdir(parents=True, exist_ok=True)

df_abundance = load_dataframe_from_file(
    input_dirpath / input_filename, index_col=[sample_key, group_key]
)

input_filename = make_filename(source_type, specie_type, "t_test_neq", filetype="tsv")
input_dirpath = EXTERNAL_DATA_DIR / "MetaboAnalyst" / "HK1-R12Ter-Statistics"
df_pvalues = load_dataframe_from_file(input_dirpath / input_filename, index_col=0)
df_pvalues = df_pvalues["p-value"]
# Plot metabolites with p < 0.1, provides some flexibility
cutoff = 0.1
species_to_plot = sorted(list(df_pvalues[df_pvalues <= cutoff].index))

for variable in species_to_plot:
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(2.5, 5))
    data = df_abundance.loc[:, [variable]]
    ax = plot_abundances(
        data=data,
        x=variable if orient == "h" else group_key,
        y=group_key if orient == "h" else variable,
        ax=ax,
        **dict(
            orient=orient,
            palette={
                group1: "#339966",
                group2: "#cc6633",
            },
            pointplot_kwargs=pointplot_kwargs,
            stripplot_kwargs=stripplot_kwargs,
            legend_kwargs=legend_kwargs,
        ),
    )

    ax.set_xlabel(group_key, fontdict=fontdict)
    ax.xaxis.grid(True, which="both")
    ax.yaxis.set_major_locator(mpl.ticker.MultipleLocator(1.5))
    ax.yaxis.set_minor_locator(mpl.ticker.MultipleLocator(0.5))
    ax.yaxis.set_tick_params(which="both", length=10)
    ax.set_ylim((-1.75, 1.75))
    ax.set_ylabel("Normalized Abundance", fontdict=fontdict)
    ax.yaxis.grid(True, which="both")
    ax.set_title(f"{variable} p={df_pvalues.loc[variable]:.4f}", fontdict=fontdict)
    sns.despine(fig)

    output_filename = make_filename(
        variable.replace("/", "-or-").replace(":", "_").replace(",", "-"), filetype=output_filetype
    )
    fig.savefig(output_dirpath / output_filename)
    plt.close()

df = df_abundance.copy()
df = df.groupby(group_key).agg(("mean", "std", "sem")).T
df.index.names = [specie_type, "Value"]
# df.columns.name = None
df = df.reset_index(drop=False)
df = df.pivot(columns=["Value"], index=[specie_type])

output_filename = make_filename(source_type, specie_type, "AbundanceStats")
save_dataframe_to_file(df, output_dirpath / output_filename, index=True)
df